# 01 - Download & Exploratory Data Analysis (EDA)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực trên tập dữ liệu COVID-19 Radiography Database.  
**Người phụ trách chính (Code Owner):** TV1 (với sự hỗ trợ kèm cặp của TV2).  

## Mục tiêu tuần 2:
1. Tự động xác định và nạp dữ liệu từ `/kaggle/input` (Dataset: `tawsifurrahman/covid19-radiography-database`).
2. Quét toàn bộ ảnh, kiểm tra tính tương ứng của mặt nạ phổi (lung mask), kích thước, chế độ màu và tính hash từng file.
3. Xuất file `manifest.csv` lưu trữ thông tin chi tiết từng ảnh để phục vụ việc phân chia dữ liệu ở Tuần 3.
4. Thống kê phân bố nhãn, đối chiếu với số liệu chuẩn của bộ dữ liệu.
5. Phân tích phân bố kích thước ảnh và độ sáng trung bình theo từng lớp; hiển thị lưới ảnh mẫu.
6. Ghi nhận các quan sát về nguy cơ học vẹt (shortcut learning) chuẩn bị cho các câu hỏi nghiên cứu sau này (RQ3 / Grad-CAM).


In [ ]:
import os
import sys
import hashlib
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm

print("Python Version:", sys.version)
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)
print("PIL Version:", Image.__version__)


In [ ]:
# 1. Định vị động thư mục dataset dưới /kaggle/input hoặc môi trường cục bộ
input_root = Path("/kaggle/input")
if not input_root.exists():
    input_root = Path(".")

print(f"Đang quét dữ liệu từ: {input_root.resolve()}")

# Tìm thư mục gốc chứa các lớp ảnh (COVID, Normal, Lung_Opacity, Viral Pneumonia)
target_root = None
candidates = [input_root] + [p for p in input_root.rglob("*") if p.is_dir()]

# Tìm thư mục chứa thư mục con 'COVID' và có 'images'
for p in candidates:
    if (p / "COVID").is_dir() and ((p / "COVID" / "images").is_dir() or (p / "Normal").is_dir()):
        target_root = p
        break

# Dự phòng nếu cấu trúc thư mục khác
if target_root is None:
    for p in candidates:
        if "radiography" in p.name.lower() and p.is_dir():
            target_root = p
            break

if target_root is None:
    target_root = input_root

print(f"Thư mục dữ liệu chính xác định được: {target_root.resolve()}")

# Lấy danh sách các thư mục lớp hợp lệ
class_folders = sorted([
    d for d in target_root.iterdir() 
    if d.is_dir() and not d.name.startswith(".") and any(
        k in d.name.lower() for k in ["covid", "normal", "lung", "opacity", "pneumonia"]
    )
])

print(f"Tìm thấy {len(class_folders)} thư mục lớp:")
for d in class_folders:
    subdirs = [s.name for s in d.iterdir() if s.is_dir()]
    print(f" - {d.name} (chứa: {subdirs})")


In [ ]:
# 2. Quét toàn bộ ảnh, kiểm tra mask, kích thước, độ sáng và tính hash
def compute_file_md5(filepath: Path, chunk_size: int = 65536) -> str:
    hasher = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

records = []
valid_exts = {'.png', '.jpg', '.jpeg'}

for class_folder in class_folders:
    class_name = class_folder.name
    images_dir = class_folder / "images"
    masks_dir = class_folder / "masks"
    
    scan_img_dir = images_dir if (images_dir.exists() and images_dir.is_dir()) else class_folder
    image_files = sorted([f for f in scan_img_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_exts])
    
    print(f"Đang quét lớp '{class_name}' ({len(image_files)} ảnh)...")
    
    for img_path in tqdm(image_files, desc=f"{class_name}"):
        img_name = img_path.name
        
        has_mask = False
        mask_path_str = ""
        rel_mask_path_str = ""
        mask_match_size = False
        
        if masks_dir.exists() and masks_dir.is_dir():
            potential_mask = masks_dir / img_name
            if potential_mask.exists():
                has_mask = True
                mask_path_str = str(potential_mask.resolve())
                try:
                    rel_mask_path_str = str(potential_mask.relative_to(target_root))
                except Exception:
                    rel_mask_path_str = f"{class_name}/masks/{img_name}"
        
        try:
            rel_img_path_str = str(img_path.relative_to(target_root))
        except Exception:
            rel_img_path_str = f"{class_name}/images/{img_name}"
        
        with Image.open(img_path) as img:
            img_w, img_h = img.size
            color_mode = img.mode
            img_arr = np.array(img)
            mean_brightness = float(np.mean(img_arr))
        
        if has_mask:
            try:
                with Image.open(mask_path_str) as m_img:
                    mw, mh = m_img.size
                    mask_match_size = (mw == img_w and mh == img_h)
            except Exception:
                mask_match_size = False
        
        file_hash = compute_file_md5(img_path)
        
        records.append({
            "image_id": img_name,
            "label": class_name,
            "image_path": str(img_path.resolve()),
            "rel_image_path": rel_img_path_str,
            "mask_path": mask_path_str,
            "rel_mask_path": rel_mask_path_str,
            "has_mask": has_mask,
            "mask_match_size": mask_match_size,
            "image_width": img_w,
            "image_height": img_h,
            "color_mode": color_mode,
            "mean_brightness": round(mean_brightness, 2),
            "file_hash": file_hash
        })

print(f"Hoàn tất quét! Tổng số bản ghi thu thập: {len(records)}")


In [ ]:
# 3. Tạo DataFrame và xuất file manifest.csv
df_manifest = pd.DataFrame(records)

working_dir = Path("/kaggle/working")
if not working_dir.exists():
    working_dir = Path(".")

manifest_out = working_dir / "manifest.csv"
df_manifest.to_csv(manifest_out, index=False)
print(f"Đã lưu manifest thành công tại: {manifest_out.resolve()} (Dung lượng: {manifest_out.stat().st_size / (1024*1024):.2f} MB)")

print("\n--- 5 DÒNG ĐẦU CỦA MANIFEST ---")
print(df_manifest.head())

print("\n--- THÔNG TIN CÁC CỘT TRONG MANIFEST ---")
print(df_manifest.info())


In [ ]:
# 4. Đối chiếu số lượng ảnh và mask từng lớp với số liệu chuẩn
known_counts = {
    "COVID": 3616,
    "Normal": 10192,
    "Lung_Opacity": 6012,
    "Viral Pneumonia": 1345
}

def find_expected(lbl: str):
    norm = lbl.lower().replace("_", " ").replace("-", " ")
    for k, v in known_counts.items():
        knorm = k.lower().replace("_", " ").replace("-", " ")
        if knorm in norm or norm in knorm:
            return v
    return None

stats_rows = []
for label, grp in df_manifest.groupby("label"):
    actual_imgs = len(grp)
    actual_masks = grp["has_mask"].sum()
    mask_match = grp["mask_match_size"].sum()
    mask_rate = (actual_masks / actual_imgs) * 100 if actual_imgs > 0 else 0
    
    expected_imgs = find_expected(label)
    delta = (actual_imgs - expected_imgs) if expected_imgs is not None else "N/A"
    
    stats_rows.append({
        "Lớp (Label)": label,
        "Số ảnh quan sát": actual_imgs,
        "Số ảnh đặc tả": expected_imgs,
        "Độ lệch (Delta)": delta,
        "Số mask quan sát": actual_masks,
        "Tỷ lệ có mask (%)": f"{mask_rate:.1f}%",
        "Mask khớp kích thước": mask_match
    })

df_stats = pd.DataFrame(stats_rows)
print("=== BẢNG ĐỐI CHIẾU SỐ LƯỢNG ẢNH VÀ MASK TỪNG LỚP ===")
print(df_stats.to_string(index=False))

total_observed = len(df_manifest)
total_expected = sum(known_counts.values())
print(f"\nTổng số ảnh quan sát được: {total_observed} | Đặc tả: {total_expected} | Độ lệch: {total_observed - total_expected}")


In [ ]:
# 5. Phân tích phân bố kích thước ảnh và độ sáng trung bình theo lớp
print("=== THỐNG KÊ KÍCH THƯỚC ẢNH THEO LỚP ===")
size_summary = df_manifest.groupby("label").agg({
    "image_width": ["min", "max", "unique"],
    "image_height": ["min", "max", "unique"],
    "color_mode": lambda s: list(s.unique()),
    "mean_brightness": ["mean", "std", "min", "max"]
})
print(size_summary)

duplicate_hashes = df_manifest[df_manifest.duplicated(subset=["file_hash"], keep=False)]
print(f"\nSố lượng ảnh có file_hash trùng lặp (exact duplicates): {len(duplicate_hashes)}")

plt.figure(figsize=(10, 5))
for label, grp in df_manifest.groupby("label"):
    plt.hist(grp["mean_brightness"], bins=40, alpha=0.5, label=f"{label} (mean={grp['mean_brightness'].mean():.1f})")
plt.title("Phân bố độ sáng trung bình của ảnh giữa 4 nhóm bệnh")
plt.xlabel("Độ sáng trung bình (Pixel Intensity [0, 255])")
plt.ylabel("Số lượng ảnh")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plot_path = working_dir / "brightness_distribution.png"
plt.savefig(plot_path, dpi=150)
print(f"Đã lưu biểu đồ phân bố độ sáng tại: {plot_path}")
plt.show()


In [ ]:
# 6. Hiển thị lưới ảnh mẫu (6 ảnh mỗi lớp kèm mask)
labels = sorted(df_manifest["label"].unique())
n_samples = 6

fig, axes = plt.subplots(len(labels), n_samples * 2, figsize=(20, 2.5 * len(labels)), squeeze=False)

for row_idx, label in enumerate(labels):
    sample_df = df_manifest[df_manifest["label"] == label].head(n_samples)
    for col_idx, (_, row) in enumerate(sample_df.iterrows()):
        img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE) if row["has_mask"] else None
        
        ax_img = axes[row_idx, col_idx * 2]
        ax_img.imshow(img, cmap="gray")
        ax_img.set_title(f"{label} #{col_idx+1}", fontsize=9)
        ax_img.axis("off")
        
        ax_mask = axes[row_idx, col_idx * 2 + 1]
        if mask is not None:
            ax_mask.imshow(mask, cmap="gray")
            ax_mask.set_title(f"Mask #{col_idx+1}", fontsize=9)
        else:
            ax_mask.text(0.5, 0.5, "No Mask", ha="center", va="center")
        ax_mask.axis("off")

plt.suptitle("Lưới mẫu ảnh X-quang và Mask phổi tương ứng (6 mẫu / lớp)", fontsize=14, y=1.01)
plt.tight_layout()
sample_grid_path = working_dir / "samples_grid.png"
plt.savefig(sample_grid_path, dpi=150, bbox_inches="tight")
print(f"Đã lưu lưới ảnh mẫu tại: {sample_grid_path}")
plt.show()


## 7. Phân tích nghi vấn Shortcut Learning tiềm ẩn (Chuẩn bị cho RQ3)

Từ các kết quả EDA sơ bộ, nhóm ghi nhận một số quan sát và nghi vấn cần theo dõi chặt chẽ trong các câu hỏi nghiên cứu sau:

1. **Sự khác biệt về nguồn gốc thu thập dữ liệu giữa các lớp:**
   - Bộ dữ liệu `COVID-19 Radiography Database` được tập hợp từ nhiều nguồn khác nhau (nguồn COVID từ các bệnh viện Ý/Tây Ban Nha, lớp Normal và Lung Opacity từ tập RSNA Pneumonia Detection Challenge).
   - Sự khác biệt về thiết bị chụp giữa các nguồn có thể tạo ra các đặc trưng giả lập (spurious correlations), chẳng hạn như: độ tương phản tổng thể, độ sáng nền ngoài lồng ngực, hoặc viền crop ảnh.
2. **Các ký hiệu nhân tạo (Watermarks, Text / L-R Markers):**
   - Quan sát lưới ảnh cho thấy một số ảnh có ký hiệu chữ cái đánh dấu chiều chụp (R/L) hoặc thông số máy nằm ở góc ngoài phổi.
   - Nếu mô hình học sâu học theo các ký hiệu này thay vì học từ tổn thương nhu mô phổi, hiện tượng "học vẹt" (shortcut learning) sẽ xảy ra, dẫn đến đánh giá sai lệch độ tin cậy trong môi trường lâm sàng thực tế.
3. **Kế hoạch kiểm chứng ở RQ3:**
   - Nhóm không đưa ra kết luận chắc chắn ở giai đoạn EDA mà đặt đây làm giả thuyết nghiên cứu chính cho **Câu hỏi nghiên cứu 3 (RQ3 - Tuần 8 & Tuần 9)**.
   - Tại Tuần 8 & 9, nhóm sẽ sử dụng kỹ thuật Grad-CAM (`07_gradcam_lung_attention.ipynb`) và đánh giá trên bộ ảnh che mặt nạ phổi (`08_rq3_masked_eval.ipynb`) để đo lường chính xác mức độ tập trung của mô hình vào trong phổi so với ngoài phổi.
